# 00 – Datenzugang (Phase 0)

Prüft für jede Datenquelle aus `PLAN.md`, dass sie lokal bzw. über Google Earth Engine **erreichbar** ist, und zählt die Kennzahlen für `DATA_SOURCES.md` direkt aus den Daten nach.
Rohdaten liegen unter `data/raw/` (nicht im Repository); Herkunft und Neu-Download stehen in `DATA_SOURCES.md`.

> **Österreich-Regel (PLAN.md §0):** Für österreichische Ertrags- und Satellitendaten werden hier nur Verfügbarkeit, Schema und Abdeckung geprüft – **keine Ertragswerte angezeigt oder ausgewertet**.

In [ ]:
import re
import zipfile
from collections import defaultdict

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from croptransfer.config import load_config, repo_path

cfg = load_config()
RAW = repo_path(cfg["paths"]["raw"])
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## 1. Google Earth Engine – Szenen über Graz und Hickory Corners (Juli 2022)

`hls_s30_unfiltered` zeigt das Footprint-Artefakt der HLS-Kacheln an der Datumsgrenze (siehe `DECISIONS.md`).

In [ ]:
from croptransfer import gee_extract

project = gee_extract.init_ee()
print("Projekt:", project)
pd.DataFrame(gee_extract.access_test(cfg)).T

## 2. KBS LTER – MCSE-Parzellenerträge (Tabelle 51)

In [ ]:
kbs = pd.read_csv(RAW / "kbs" / "kbs_datatable_51.csv", comment="#")
maize = kbs[kbs.Crop.str.contains("Zea", na=False) & kbs.whole_plot_yield_kg_ha.notna()].copy()
maize["plot_id"] = maize.Treatment + maize.Replicate
print("Mais-Parzellen-Jahre gesamt:", len(maize), "| Mais-Jahre:", maize.Year.nunique())
per_year = maize.groupby("Year").agg(n=("plot_id", "size"),
                                     mean_t_ha=("whole_plot_yield_kg_ha", lambda s: s.mean() / 1000))
display(per_year.round(2).T)
hls = maize[maize.Year >= cfg["years"]["groundtruth"][0]]
print("HLS-Zeit:", len(hls), "Parzellen-Jahre in", sorted(hls.Year.unique()),
      "| Parzellen:", hls["plot_id"].nunique())

**Ertragstrend und RY-Normalwert** (Datenkritik §3.1 b): Der Normalwert für RY hängt davon ab, welche „anderen Jahre“ eingehen.

In [ ]:
import numpy as np

slope = np.polyfit(per_year.index, per_year.mean_t_ha, 1)[0]
print(f"Linearer Trend über alle Mais-Jahre: {slope:+.3f} t/ha pro Jahr")

def relative_yield(df, pool_years):
    rows = []
    for (p, y), g in df.groupby(["plot_id", "Year"]):
        other = df[(df["plot_id"] == p) & (df.Year != y) & df.Year.isin(pool_years)]
        if other.Year.nunique() >= cfg["groundtruth"]["ry_min_other_years"]:
            rows.append({"plot": p, "Year": y, "Treatment": p[:2],
                         "RY": g.whole_plot_yield_kg_ha.mean() / other.whole_plot_yield_kg_ha.mean()})
    return pd.DataFrame(rows)

hls_years = sorted(hls.Year.unique())
thr = cfg["targets"]["at_risk_threshold"]
for name, pool in [("alle Mais-Jahre", maize.Year.unique()), ("nur HLS-Jahre", hls_years)]:
    r = relative_yield(maize, pool).query("Year in @hls_years")
    risk = r[r.RY < thr]
    print(f"{name:16s}: Median-RY je Jahr {r.groupby('Year').RY.median().round(2).to_dict()} | "
          f"RY<{thr}: {len(risk)} (je Jahr {risk.Year.value_counts().sort_index().to_dict()}, "
          f"je Behandlung {risk.Treatment.value_counts().sort_index().to_dict()})")

## 3. USDA BARC – Kriging-Ertragskarten (Mais)

In [ ]:
barc_fy = defaultdict(set)
for zpath in sorted((RAW / "barc").glob("20*.zip")):
    year = int(zpath.stem)
    for name in zipfile.ZipFile(zpath).namelist():
        m = re.match(rf"{year}/6A[^/]*/(.+)_{year}_corn_clip\.tif$", name, re.I)
        if m:  # normalise naming variants (USDA_ prefix, Central_Farm vs CentralFarm)
            key = m.group(1).lower().replace("usda_", "").replace("central_farm", "centralfarm")
            barc_fy[key].add(year)
n_years = pd.Series({k: len(v) for k, v in barc_fy.items()})
print("Mais-Feld-Jahre:", n_years.sum(), "| Felder (Namen normalisiert):", len(n_years))
print("Felder nach Anzahl Mais-Jahre:", n_years.value_counts().sort_index().to_dict())
print("Feld-Jahre mit definiertem RY (Feld hat >= 3 Mais-Jahre):", n_years[n_years >= 3].sum())
pd.Series({y: sum(y in v for v in barc_fy.values()) for y in range(2014, 2022)}, name="Mais-Karten").to_frame().T

## 4. Colorado (Akron) – Ertragsstichproben

In [ ]:
colo = pd.read_csv(RAW / "colorado" / "Modeling_data.csv")
cm = colo[colo.crop.str.lower().str.contains("corn|maize")]
units = cm.groupby(["field", "year"]).agg(n_points=("yield_kg_ha", "size"),
                                          mean_t_ha=("yield_kg_ha", lambda s: s.mean() / 1000))
display(units.round(2).T)
print("Mais-Jahre je Teilfläche:", cm.groupby("field").year.nunique().value_counts().to_dict(),
      "-> RY (>= 3 Mais-Jahre) nirgends definiert")

## 5. Mead (AmeriFlux) – Status

In [ ]:
print((RAW / "mead" / "STATUS.txt").read_text(encoding="utf-8"))

## 6. Ergänzende Quellen: One Acre Fund, YieldSAT, SSURGO

### 6a. One Acre Fund (Zenodo) – Kennzahlen und Vorab-Filter (PLAN §3.1)

Es werden keine Satellitendaten angesehen.

In [ ]:
from croptransfer import one_acre_fund

oaf = one_acre_fund.summary(one_acre_fund.load(), cfg)
display(pd.Series(oaf["facts"], name="Wert").to_frame())
display(oaf["by_year"])
oaf["exact_by_year"]

### 6b. YieldSAT

Zugang beantragt der Projektinhaber über das Formular auf yieldsat.github.io. Bis dahin sind keine Daten lokal vorhanden (Status in `DATA_SOURCES.md` §10).

### 6c. SSURGO (Soil Data Access, ohne Login) – Testabfrage KBS-Parzellen und BARC 1-7

In [ ]:
from croptransfer import soil_ssurgo

ssurgo_rows = []
for name, wkt in soil_ssurgo._test_fields(cfg).items():
    vals, _ = soil_ssurgo.query_field(wkt, cfg)
    ssurgo_rows.append({"Feld": name, **vals})
ssurgo_table = pd.DataFrame(ssurgo_rows).set_index("Feld")
print("NCCPI Mais je Feld (eindeutige Werte):", ssurgo_table.nccpi_corn.round(3).unique())
ssurgo_table.round(3).tail(3)

## 7. INVEKOS Schläge (AMA) – nur Schema und Zählung

Österreich-Regel: Es werden nur Dateien, Spalten und Anzahlen geprüft. Die Steiermark-Zuordnung erfolgt über den repräsentativen Punkt jedes Schlags in den Gemeinden (Stand 2025-01-01).

In [ ]:
from croptransfer import invekos

gpkgs = invekos.find_gpkgs(RAW / "invekos")
schemas = {y: invekos.schema(p) for y, p in gpkgs.items()}
pd.DataFrame({y: {"Datei": gpkgs[y].name, "Schläge": s["features"],
                  "CRS": str(s["crs"])[:40], "Spalten": len(s["columns"]),
                  "SNAR_CODE": "SNAR_CODE" in s["columns"],
                  "SNAR_BEZEICHNUNG": "SNAR_BEZEICHNUNG" in s["columns"]}
              for y, s in schemas.items()}).T

In [ ]:
cols = {y: set(s["columns"]) for y, s in schemas.items()}
common = set.intersection(*cols.values())
print("Gemeinsame Spalten:", sorted(common))
for y, c in cols.items():
    if c - common:
        print(f"{y}: zusätzlich {sorted(c - common)}")

In [ ]:
codes = {"105": "Körnermais", "106": "CCM", "109": "Silomais", "173": "Saatmaisvermehrung",
         "175": "Wintergerste/Körnermais"}
gem = gpd.read_file(next((RAW / "admin_at" / "GEM_20250101").glob("*.shp")))
rows = []
for y, p in gpkgs.items():
    c = invekos.crop_code_counts(p, codes)
    st = invekos.count_in_state(p, gem, code="105", min_area_ha=cfg["field"]["min_area_ha"])
    rows.append({"Jahr": y, **{f"n {codes[k]}": int(c.loc[k, "n"]) for k in codes},
                 "Körnermais Stmk n": st["n"], f"Stmk ≥ {cfg['field']['min_area_ha']} ha": st["n_ge_min_area"],
                 "ungültige Geom. (Körnermais AT)": st["invalid_geometries"], "ohne Gemeinde": st["unmatched"]})
invekos_counts = pd.DataFrame(rows).set_index("Jahr")
invekos_counts

## 8. Verwaltungsgrenzen Österreich (Statistik Austria)

In [ ]:
admin = {p.name: next(p.glob("*.shp")) for p in sorted((RAW / "admin_at").iterdir()) if p.is_dir()}
fig, axes = plt.subplots(1, len(admin), figsize=(4 * len(admin), 3))
for ax, (name, shp) in zip(axes, admin.items()):
    g = gpd.read_file(shp)
    g.boundary.plot(ax=ax, linewidth=0.3, color="k")
    ax.set_title(f"{name}\n{len(g)} Einheiten, EPSG:{g.crs.to_epsg()}, ungültig: {(~g.is_valid).sum()}", fontsize=9)
    ax.set_axis_off()
plt.tight_layout()

## 9. CY-Bench v1.10 – Mais

EU-Länder mit Abdeckung; für Österreich nur Schema und Abdeckung (Österreich-Regel).

In [ ]:
cyb_dir = RAW / "cybench" / "cybench-data" / "maize"
eu = []
for d in sorted(cyb_dir.iterdir()):
    y = pd.read_csv(d / f"yield_maize_{d.name}.csv")
    pos = y[y["yield"] > 0]
    eu.append({"Land": d.name, "Regionen": y.adm_id.nunique(),
               "Jahre": f"{pos.harvest_year.min()}–{pos.harvest_year.max()}" if len(pos) else "–",
               "Region-Jahre (Ertrag>0)": len(pos), "Nullerträge": int((y["yield"] == 0).sum())})
pd.DataFrame(eu).set_index("Land").T

In [ ]:
at_files = sorted(p.name for p in (cyb_dir / "AT").glob("*.csv"))
cyb_at = pd.read_csv(cyb_dir / "AT" / "yield_maize_AT.csv")
print("AT-Dateien:", at_files)
print("AT-Ertrag: Spalten", list(cyb_at.columns), "| Regionen", sorted(cyb_at.adm_id.unique()),
      "| Jahre", cyb_at.harvest_year.min(), "-", cyb_at.harvest_year.max(),
      "| Zeilen", len(cyb_at), "| fehlend", int(cyb_at["yield"].isna().sum()))

## 10. Statistik Austria – `stat_at_maize.csv` (nur Schema und Abdeckung)

In [ ]:
stat = pd.read_csv(RAW / "stat_at_maize.csv")
print("Spalten:", list(stat.columns), "| Zeilen:", len(stat))
display(pd.crosstab(stat.year, stat.status).T)
print("fehlende Erträge:", int(stat.yield_t_ha.isna().sum()),
      "| Bundesländer:", stat.bundesland.nunique())

## 11. Österreichische Versuchsberichte – `at_trials.csv` (nur Bestand, keine Erträge)

In [ ]:
trials = pd.read_csv(RAW / "at_trials.csv")
print("Spalten:", list(trials.columns), "| Zeilen:", len(trials))
trials.groupby(["bundesland", "site"]).year.agg(["count", "min", "max"])

## Zusammenfassung

Status je Quelle und Befunde: `DATA_SOURCES.md` (Übersicht) und `reports/data_review/phase0_data_access.md`.